In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType, IntegerType
from pyspark.sql.functions import col, regexp_replace, nullif, lit, to_date
from pyspark.sql.types import DecimalType

# def clean_decimal(c, p=38, s=8):
#     cleaned = regexp_replace(col(c), "[^0-9]", "")
#     return round(nullif(cleaned, lit("")),5).alias(c)
def clean_decimal(c, p=38, s=12):
    return round(
        when(col(c) == "", None)
        .otherwise(col(c).cast("double")) 
        .cast(DecimalType(p, s)),
        5
    ).alias(c)
def clean_int(c):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return when(
        length(cleaned) == 0,
        None
    ).otherwise(
        cleaned.cast(IntegerType())
    ).alias(c)
def clean_short(c):
    cleaned = regexp_replace(col(c), "[^0-9]", "")
    return when(
        length(cleaned) == 0,
        None
    ).otherwise(
        cleaned.cast("short")
    ).alias(c)
def clean_date(c):
    return coalesce(
        expr(f"try_to_date({c}, 'M/d/yyyy')"),
        expr(f"try_to_date({c}, 'MM/dd/yyyy')"),
        expr(f"try_to_date({c}, 'yyyy-MM-dd')"),
        expr(f"try_to_date({c}, 'yyyyMMdd')")
    ).alias(c)
#df_src = spark.read.table("mb_poc.data_raw.cib")
df_src = (
    spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .csv("/Volumes/mb_poc/data_raw/uc2/PTKHDN_RPT_CIB_QTC_M01_TOI_20250531_v2.csv")
)

df_tgt = (
    df_src.select(
        clean_date("TXN_DT"),
        col("CST_NBR").alias("CST_NBR"),
        col("CST_NM").alias("CST_NM"),
        col("BR_CODE").alias("BR_CODE"),
        col("BR_NM").alias("BR_NM"),
        col("RGON").alias("RGON"),
        col("CST_BR_CODE").alias("CST_BR_CODE"),
        col("CST_BR_NM").alias("CST_BR_NM"),
        col("CST_RGON").alias("CST_RGON"),
        col("CST_LOB").alias("CST_LOB"),
        col("HUB_F").alias("HUB_F"),
        col("ACT_F").alias("ACT_F"),
        col("NEW_ACT_F").alias("NEW_ACT_F"),
        clean_date("CRT_DT"),
        col("CST_PERF_ST").alias("CST_PERF_ST"),
        col("CR_PERF_ST").alias("CR_PERF_ST"),
        col("CST_GRP").alias("CST_GRP"),
        clean_decimal("VND_SHRT_LOAN_BAL"),
        clean_decimal("VND_LN_LOAN_BAL"),
        clean_decimal("USD_SHRT_LOAN_BAL"),
        clean_decimal("USD_LN_LOAN_BAL"),
        clean_decimal("VND_SHRT_LOAN_BAL_MBV"),
        clean_decimal("VND_LN_LOAN_BAL_MBV"),
        clean_decimal("USD_SHRT_LOAN_BAL_MBV"),
        clean_decimal("USD_LN_LOAN_BAL_MBV"),
        clean_decimal("VND_AVG_SHRT_LOAN_BAL_YTD"),
        clean_decimal("VND_AVG_LN_LOAN_BAL_YTD"),
        clean_decimal("USD_AVG_SHRT_LOAN_BAL_YTD"),
        clean_decimal("USD_AVG_LN_LOAN_BAL_YTD"),
        clean_decimal("VND_CASA_BAL"),
        clean_decimal("VND_TERM_FD_BAL"),
        clean_decimal("USD_CASA_BAL"),
        clean_decimal("USD_TERM_FD_BAL"),
        clean_decimal("VND_AVG_CASA_YTD"),
        clean_decimal("VND_AVG_TERM_FD_YTD"),
        clean_decimal("USD_AVG_CASA_YTD"),
        clean_decimal("USD_AVG_TERM_FD_YTD"),
        clean_decimal("BOND_BAL"),
        clean_decimal("AVG_BOND_BAL_YTD"),
        clean_decimal("GNT_BAL"),
        clean_decimal("AVG_GNT_BAL_YTD"),
        clean_decimal("ARS_BAL"),
        clean_decimal("NPL_BAL"),
        clean_decimal("ARS_BAL_MBV"),
        clean_decimal("NPL_BAL_MBV"),
        clean_decimal("UP_BAL"),
        clean_decimal("AVG_UP_BAL_YTD"),
        clean_decimal("ENTRST_BAL"),
        clean_decimal("MB_OB_ENTRST_BAL"),
        clean_decimal("OB_MB_ENTRST_BAL"),
        clean_decimal("SALE_TRD_FNC_YTD"),
        clean_decimal("SALE_FX_YTD"),
        clean_decimal("OFF_BSH_LOAN_BAL"),
        clean_decimal("PKG_LOAN_BAL"),
        clean_decimal("DSBR_AMT_YTD"),
        clean_decimal("VND_UNTERM_CPTL_SALE_PFT"),
        clean_decimal("VND_TERM_CPTL_SALE_PFT"),
        clean_decimal("USD_UNTERM_CPTL_SALE_PFT"),
        clean_decimal("USD_TERM_CPTL_SALE_PFT"),
        clean_decimal("VND_INT_REAL_YTD"),
        clean_decimal("VND_TERM_INT_REAL_YTD"),
        clean_decimal("USD_INT_REAL_YTD"),
        clean_decimal("USD_TERM_INT_REAL_YTD"),
        #lit(0).alias("AVG_NIM_VND_CASA_YTD"),
        #lit(0).alias("AVG_NIM_TERM_VND_DEP_YTD"),
        #lit(0).alias("AVG_NIM_USD_CASA_YTD"),
        #lit(0).alias("AVG_NIM_TERM_USD_DEP_YTD"),
        clean_decimal("VND_SHRT_TERM_CPTL_PRCH_INT"),
        clean_decimal("VND_LN_TERM_CPTL_PRCH_INT"),
        clean_decimal("USD_SHRT_TERM_CPTL_PRCH_INT"),
        clean_decimal("USD_LN_TERM_CPTL_PRCH_INT"),
        clean_decimal("VND_SHRT_TERM_CR_INT_RCV"),
        clean_decimal("VND_LN_TERM_CR_INT_RCV"),
        clean_decimal("USD_SHRT_TERM_CR_INT_RCV"),
        clean_decimal("USD_LN_TERM_CR_INT_RCV"),
        clean_decimal("ON_BSH_HANGING_INT_REV"),
        clean_decimal("OFF_BSH_REV"),
        clean_decimal("OFF_BSH_INT_REV"),
        clean_decimal("ON_BSH_REV"),
        clean_decimal("NET_REV_BFR_RSK"),
        clean_decimal("DEBT_EXTRA_ORIG"),
        clean_decimal("NET_PFT_REV"),
        clean_decimal("PRVN_EXPN"),
        clean_decimal("NET_REV_BFR_RSK_NRL"),
        clean_decimal("AVG_BAL_LCY"),
        clean_decimal("GNT_REV"),
        clean_decimal("OTH_EXTRA_ORIG"),
        clean_decimal("FX_SVC_REV"),
        clean_decimal("IB_SVC_REV"),
        clean_decimal("FX_REV"),
        clean_decimal("NII"),
        clean_decimal("BOND_INT_RCV"),
        clean_decimal("INTER_PYMT_SVC"),
        clean_decimal("CR_INT_RCV"),
        clean_decimal("CR_FTP_EXPN"),
        clean_decimal("DOM_PYMT_SVC"),
        clean_decimal("CST_INT_RCV"),
        clean_decimal("BOND_FTP_EXPN"),
        clean_decimal("DEP_FTP_RCV"),
        clean_decimal("DEP_INT_EXPN"),
        clean_decimal("NET_PRVN_RCV"),
        clean_decimal("IVS_BNK_SVC"),
        clean_decimal("IVS_RCV"),
        clean_decimal("OPRN_EXPN")

    )
)
#df_tgt.printSchema()
#df_tgt.display()
#df_tgt.explain(True)
#spark.table("mb_poc.gold.rpt_cib_qtc_m01_toi").printSchema()

df_tgt.write.mode("append").insertInto("mb_poc.gold.rpt_cib_qtc_m01_toi")

